### Recommendation System

Data Description:

Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.
Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset. 
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.
Analyze the performance of the recommendation system and identify areas of improvement.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

### 1. Import Libraries

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(7, 5)
plt.rcParams['figure.dpi']=250
sns.set_theme(style='darkgrid', palette='viridis')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

### 2. Load Dataset & Preprocessing

In [5]:
data = pd.read_csv("C:\\Users\\bilad\\ExcelR\\Assignments\\Datasets\\anime.csv")   # change filename if needed
data.head()


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [6]:
data.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [7]:
data.nunique()

anime_id    12294
name        12292
genre        3264
type            6
episodes      187
rating        598
members      6706
dtype: int64

In [8]:
data.duplicated().sum()

0

In [9]:
#checking for missing values
data.isnull().sum()


anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [10]:

# Fill missing genres, type with 'Unknown'
data['genre'] = data['genre'].fillna('Unknown')
data['type']= data['type'].fillna('unknown')

#Fill missing values in rating with its mean
data['rating']=data['rating'].fillna(data['rating'].mean())


In [11]:
data.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [12]:
# Data type changing 
data['episodes']=pd.to_numeric(data['episodes'], errors='coerce').astype('Int64')

sns.countplot(y=data['type'])
plt.show()

### 3. Feature Engineering

In [15]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['genre'])


### 4. Cosine Similarity Matrix

In [17]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)


In [18]:
data['name'].duplicated().sum()

2

### 5. Create Recommendation Function

In [20]:
# Reset index to safely map anime names
data = data.reset_index()
indices = pd.Series(data.index, index=data['name'])

def recommend_anime(title, df, cosine_sim, num_recommendations=5, threshold=0.3):
    if title not in indices:
        return "Anime not found in dataset!"

    idx = indices[title]                 # index of target anime
    sim_scores = list(enumerate(cosine_sim[idx]))   # similarity with others
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    anime_indices = [i[0] for i in sim_scores]

    return data[['name', 'genre', 'rating']].iloc[anime_indices]


### 6. Try Recommendations

In [22]:
recommend_anime("Naruto", data, cosine_sim, threshold=0.2)  # example

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
841,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",7.81
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53


In [23]:
recommend_anime("Naruto", data, cosine_sim, threshold=0.4)  # example

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
841,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",7.81
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53


In [24]:
recommend_anime("Naruto", data, cosine_sim, threshold=0.7)  # example

,name,genre,rating
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",7.94
841,Naruto,"Action, Comedy, Martial Arts, Shounen, Super P...",7.81
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",7.68
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",7.58
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",7.53


### 7. Improved Version (Add Episodes + Rating + Members)

In [26]:
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

data['episodes']=data['episodes'].fillna(0)

# Normalize numerical features
scaler = MinMaxScaler()
num_features = scaler.fit_transform(data[['episodes', 'rating', 'members']])

# Combine genre TF-IDF + Numerical Features
features = hstack([tfidf_matrix, num_features])

cosine_sim2 = cosine_similarity(features, features)

def recommend_anime_improved(title, num_recommendations=5, threshold=0.4):
    if title not in indices:
        return "Anime not found!"

    idx = indices[title]
    sim_scores = sorted(list(enumerate(cosine_sim2[idx])), key=lambda x: x[1], reverse=True)[1:num_recommendations+1]
    anime_indices = [i[0] for i in sim_scores]

    return data[['name', 'genre', 'rating', 'episodes']].iloc[anime_indices]


### 8. Try Improved Recommendations

In [28]:
recommend_anime_improved("One Piece", 5, threshold=0.4)


,name,genre,rating,episodes
241,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.27,1
86,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",8.54,25
231,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.29,1
896,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.78,1
6,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",9.13,148


In [29]:
recommend_anime_improved("One Piece", 5, threshold=0.6)


,name,genre,rating,episodes
241,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.27,1
86,Shingeki no Kyojin,"Action, Drama, Fantasy, Shounen, Super Power",8.54,25
231,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.29,1
896,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.78,1
6,Hunter x Hunter (2011),"Action, Adventure, Shounen, Super Power",9.13,148


### 9. Save Function for Deployment

In [31]:
def get_recommendations(title, improved=False):
    if improved:
        return recommend_anime_improved(title, 5)
    else:
        return recommend_anime(title, 5)


### Threshold Experimentation

Threshold Value	Effect
- Low (0.2)-> More recommendations, but may include weakly related anime
- Medium (0.4)->	Balanced results (good relevance + quantity)
- High (0.6 )  -> 	Fewer but very highly similar anime

### Interview Questions

#### 1. Can you explain the difference between user-based and item-based collaborative filtering?
1. User-Based Collaborative Filtering
   - Focuses on finding similar users based on their preferences or ratings.
   - Recommends items that similar users have liked but the target user has not seen.
   - Similarity is calculated between users (user–user similarity).
   - Performance can degrade when the number of users is very large.
   - User preferences may change frequently, making it less stable.

2. Item-Based Collaborative Filtering
   - Focuses on finding similar items based on user interactions.
   - Recommends items similar to those the user already likes.
   - Similarity is calculated between items (item–item similarity).
   - More scalable and efficient for large datasets.
   - Item relationships are more stable over time.

#### 2. What Is Collaborative Filtering and How Does It Work?
- Collaborative Filtering (CF) is a recommendation technique that predicts a user’s preferences by analyzing patterns of user behavior, such as ratings, likes, or purchases, rather than relying on item content. <BR>
#### How Does It Work <BR>
Step 1: Collect User–Item Interactions
- A matrix is created where:
- Rows → Users
- Columns → Items
- Values → Ratings or interactions

Step 2: Compute Similarity
- Similarity is calculated using:
- Cosine Similarity
- Pearson Correlation
- Euclidean Distance

Step 3: Generate Recommendations
- Predict missing ratings
- Recommend items with highest predicted scores matrix.
